# Weibull Survival Analysis — Customer LTV & Opportunity Cost
### Dataset: Olist Brazilian E-Commerce (Kaggle, 2016–2018)

---

> **Central question:** Is customer segmentation a data problem or a marketing problem?
>
> This notebook uses Weibull survival analysis to bridge
> **past behavior → future probability → economic decision**.

**Pipeline:**
1. Fit Weibull on average inter-purchase intervals of repeat buyers (`media_dias`)
2. Apply fitted parameters to score **all** customers via conditional probability
3. Translate probability into opportunity cost and CRM segments


## Section 0 — Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mtick
from scipy.stats import weibull_min
from scipy.special import gamma as gamma_fn

os.makedirs("plots", exist_ok=True)

# ── Global parameters ──────────────────────────────────────────────────────
HORIZON         = 30    # decision window (days)
MARGIN          = 0.35  # net profit margin
ACTIVATION_COST = 8.0   # R$ cost to reach one customer

# ── Plot style — matches ltv notebooks ────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor" : "#F8FAFC",
    "axes.facecolor"   : "#F8FAFC",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "grid.linestyle"   : "--",
    "font.family"      : "DejaVu Sans",
})

PALETTE = {
    "Champions" : "#10B981",
    "Nurture"   : "#3B82F6",
    "At Risk"   : "#F59E0B",
    "Dormant"   : "#94A3B8",
    "accent"    : "#EF4444",
    "dark"      : "#1E293B",
}

print(f"Parameters: HORIZON={HORIZON}d | MARGIN={MARGIN:.0%} | ACTIVATION_COST=R${ACTIVATION_COST}")


In [ ]:
import kagglehub
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Dataset path: {path}")

orders    = pd.read_csv(f"{path}/olist_orders_dataset.csv")
customers = pd.read_csv(f"{path}/olist_customers_dataset.csv")
payments  = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")

print(f"orders:    {orders.shape}")
print(f"customers: {customers.shape}")
print(f"payments:  {payments.shape}")


In [ ]:
# ── Clean & merge ──────────────────────────────────────────────────────────
orders_clean = orders[orders["order_status"] == "delivered"].copy()
orders_clean["order_purchase_timestamp"] = pd.to_datetime(
    orders_clean["order_purchase_timestamp"]
)
orders_clean = orders_clean.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id"
)

# Revenue: exclude undefined payment types
revenue = (
    payments[payments["payment_type"] != "not_defined"]
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
    .rename(columns={"payment_value": "order_value"})
)

df = (
    orders_clean[["order_id", "customer_unique_id", "order_purchase_timestamp"]]
    .merge(revenue, on="order_id")
    .rename(columns={
        "customer_unique_id"       : "customer",
        "order_purchase_timestamp" : "purchase_date",
    })
)
df = df.sort_values(["customer", "purchase_date"])

print(f"Clean dataset:    {df.shape}")
print(f"Unique customers: {df['customer'].nunique():,}")
print(f"Date range:       {df['purchase_date'].min().date()} → {df['purchase_date'].max().date()}")


## Section 1 — Feature Engineering: `media_dias`

The key variable is the **average inter-purchase interval** (`media_dias`) per customer.

**Why aggregate to customer-level averages before fitting?**
Using `media_dias` puts one data point per repeat buyer in the likelihood function — this
models the distribution of *customer behaviour types* rather than the distribution of individual events.
It avoids giving disproportionate weight to customers with many orders.

> **Scoring scope:** Weibull parameters are estimated from repeat buyers only.
> The fitted parameters are then applied as a **population prior** to score all customers,
> including one-time buyers, based on their current `days_silent`.


In [ ]:
# ── Inter-purchase intervals ────────────────────────────────────────────────
df["interval_days"] = df.groupby("customer")["purchase_date"].diff().dt.days

# Repeat buyers: at least 2 purchases → at least 1 observed interval
repeat_buyers = (
    df.dropna(subset=["interval_days"])
    .groupby("customer")
    .agg(
        media_dias  = ("interval_days", "mean"),
        n_purchases = ("order_id",      "count"),
        avg_ticket  = ("order_value",   "mean"),
        total_spent = ("order_value",   "sum"),
    )
    .reset_index()
)
repeat_buyers = repeat_buyers[repeat_buyers["media_dias"] > 0]

n_total  = df["customer"].nunique()
n_repeat = len(repeat_buyers)

print("=" * 55)
print("CUSTOMER BASE SUMMARY")
print("=" * 55)
print(f"Total unique customers:      {n_total:>8,}")
print(f"Repeat buyers (fit base):    {n_repeat:>8,}  ({100*n_repeat/n_total:.1f}%)")
print(f"One-time buyers (scored):    {n_total-n_repeat:>8,}  ({100*(1-n_repeat/n_total):.1f}%)")
print()
print("media_dias (avg inter-purchase interval) distribution:")
print(repeat_buyers["media_dias"].describe().round(1).to_string())


## Section 2 — Weibull Distribution: Mathematical Derivation & MLE Fit

### 2.1 Why Weibull?

The exponential distribution assumes a **constant hazard rate** — repurchase probability per unit
time never changes regardless of how long the customer has been silent. This is rarely true in retail.

The **Weibull distribution** generalises this with shape parameter $\beta$:

### 2.2 Core Functions

$$f(t) = \frac{\beta}{\lambda}\left(\frac{t}{\lambda}\right)^{\!\beta-1}\exp\!\left[-\left(\frac{t}{\lambda}\right)^\beta\right]$$

| Function | Formula | Meaning |
|---|---|---|
| CDF | $F(t) = 1 - e^{-(t/\lambda)^\beta}$ | P(repurchase by day $t$) |
| Survival | $S(t) = e^{-(t/\lambda)^\beta}$ | P(still inactive at day $t$) |
| Hazard | $h(t) = (\beta/\lambda)(t/\lambda)^{\beta-1}$ | Instantaneous repurchase rate |

### 2.3 Expected Value

$$\mathbb{E}[T] = \lambda\cdot\Gamma\!\left(1 + \frac{1}{\beta}\right)$$

### 2.4 Shape parameter $\beta$

| $\beta$ | Hazard | CRM implication |
|---|---|---|
| $< 1$ | Decreasing | Early window is critical — act fast |
| $= 1$ | Constant | Memoryless (Exponential) |
| $> 1$ | Increasing | Repurchase builds over time |

### 2.5 MLE via `scipy.stats.weibull_min`

`weibull_min(c, loc, scale)` with `floc=0` → $c = \beta$, $\text{scale} = \lambda$.


In [ ]:
x = repeat_buyers["media_dias"].values

shape, loc, scale = weibull_min.fit(x, floc=0)

theoretical_mean = scale * gamma_fn(1 + 1 / shape)

print("=" * 50)
print("WEIBULL MLE PARAMETERS")
print("=" * 50)
print(f"  β (shape):      {shape:.4f}")
print(f"  λ (scale):      {scale:.2f} days")
print(f"  E[T] fitted:    {theoretical_mean:.1f} days")
print(f"  E[T] observed:  {x.mean():.1f} days")
print()

if shape < 1:
    print(f"  β < 1 → DECREASING hazard")
    print(f"  Customers who repurchase do so early.")
    print(f"  The longer the silence, the lower the probability of return.")
elif abs(shape - 1) < 0.05:
    print(f"  β ≈ 1 → approximately CONSTANT hazard (Exponential)")
else:
    print(f"  β > 1 → INCREASING hazard")
    print(f"  Repurchase probability builds over time.")

# KS test
from scipy import stats as _stats
ks_stat, ks_pval = _stats.kstest(x, "weibull_min", args=(shape, 0, scale))
print()
print(f"  KS test: stat={ks_stat:.4f}  p={ks_pval:.4f}")


## Section 3 — The Bridge: Past → Future

### Conditional Repurchase Probability

$$
P\bigl(T \leq t_0 + \Delta t \mid T > t_0\bigr)
= \frac{F(t_0 + \Delta t) - F(t_0)}{S(t_0)}
$$

Each customer enters this formula with their own $t_0$ (days since last purchase),
yielding a personalised repurchase probability for the next 30 days.


In [ ]:
def conditional_prob(t0, delta_t, shape=shape, scale=scale):
    """
    P(repurchase within delta_t days | already t0 days silent).
    Parameters are explicit so the function is reusable for different fits.
    """
    t0      = np.asarray(t0,      dtype=float)
    delta_t = np.asarray(delta_t, dtype=float)
    F_t0       = weibull_min.cdf(t0,           shape, loc=0, scale=scale)
    F_t0_delta = weibull_min.cdf(t0 + delta_t, shape, loc=0, scale=scale)
    S_t0       = 1.0 - F_t0
    with np.errstate(divide="ignore", invalid="ignore"):
        prob = np.where(S_t0 > 1e-9, (F_t0_delta - F_t0) / S_t0, 0.0)
    return prob

print(f"P(repurchase within {HORIZON}d | t0 days silent)")
print(f"  β = {shape:.4f}   λ = {scale:.2f} days")
print("-" * 38)
print(f"  {'t0':>12} | {'P(repurchase)':>14}")
print("-" * 38)
for t0 in [7, 14, 30, 60, 90, 120, 180, 270, 365]:
    p = float(conditional_prob(t0, HORIZON))
    print(f"  {t0:>12} | {p:>13.1%}")


## Section 4 — Score All Customers & Opportunity Cost

The Weibull parameters (estimated from repeat buyers) are used as a **population prior**
and applied to every customer — including one-time buyers.

$$
\text{Expected Net}_i = P(\text{repurchase})_i \times \bar{\text{ticket}}_i \times m
$$
$$
\text{Opportunity Cost}_i = \text{Expected Net}_i - C_a
$$
$$
\text{ROI}_i = (\text{Expected Net}_i - C_a) / C_a
$$


In [ ]:
REF_DATE = df["purchase_date"].max()
print(f"Reference date: {REF_DATE.date()}")

# ── All customers summary ──────────────────────────────────────────────────
all_customers = (
    df.groupby("customer")
    .agg(
        last_purchase = ("purchase_date", "max"),
        avg_ticket    = ("order_value",   "mean"),
        total_spent   = ("order_value",   "sum"),
        n_orders      = ("order_id",      "count"),
    )
    .reset_index()
)
all_customers["days_silent"] = (
    REF_DATE - all_customers["last_purchase"]
).dt.days.clip(lower=1)

# ── Score: conditional probability ─────────────────────────────────────────
all_customers["prob_30d"] = conditional_prob(
    all_customers["days_silent"].values, HORIZON
)

# ── Economic framework ─────────────────────────────────────────────────────
all_customers["expected_revenue"]   = all_customers["prob_30d"] * all_customers["avg_ticket"]
all_customers["expected_net_value"] = all_customers["expected_revenue"] * MARGIN - ACTIVATION_COST
all_customers["opportunity_cost"]   = (all_customers["expected_revenue"] * MARGIN).clip(lower=0)
all_customers["roi_activation"]     = (
    (all_customers["expected_revenue"] * MARGIN - ACTIVATION_COST) / ACTIVATION_COST
).replace([np.inf, -np.inf], np.nan)

n_pos = (all_customers["roi_activation"] > 0).sum()
total_oc = all_customers["opportunity_cost"].sum()

print(f"Customers scored:              {len(all_customers):,}")
print(f"  of which repeat buyers:      {n_repeat:,}")
print(f"  of which one-time buyers:    {len(all_customers) - n_repeat:,}")
print(f"Positive ROI customers:        {n_pos:,}  ({100*n_pos/len(all_customers):.1f}%)")
print(f"Total 30d opportunity cost:    R$ {total_oc:,.0f}")


## Section 5 — 2×2 CRM Segmentation

| | High Ticket | Low Ticket |
|---|---|---|
| **High P(repurchase)** | 🎯 Champions | 📢 Nurture |
| **Low P(repurchase)**  | ⚠️ At Risk  | 💤 Dormant |


In [ ]:
prob_med   = all_customers["prob_30d"].median()
ticket_med = all_customers["avg_ticket"].median()

print(f"Probability threshold (median): {prob_med:.4f}")
print(f"Ticket threshold (median):      R$ {ticket_med:.2f}")

def assign_segment(row):
    hp = row["prob_30d"]   >= prob_med
    ht = row["avg_ticket"] >= ticket_med
    if   hp and     ht: return "Champions"
    elif hp and not ht: return "Nurture"
    elif not hp and ht: return "At Risk"
    else:               return "Dormant"

ACTIONS = {
    "Champions": "Loyalty program + upsell",
    "At Risk":   "Urgent win-back offer (30d window)",
    "Nurture":   "Cross-sell to grow basket size",
    "Dormant":   "Low-cost drip or reallocate budget",
}

all_customers["segment"] = all_customers.apply(assign_segment, axis=1)
all_customers["action"]  = all_customers["segment"].map(ACTIONS)

seg_summary = (
    all_customers.groupby("segment")
    .agg(
        n_customers    = ("customer",         "count"),
        avg_prob_30d   = ("prob_30d",         "mean"),
        avg_ticket     = ("avg_ticket",       "mean"),
        total_opp_cost = ("opportunity_cost", "sum"),
        avg_roi        = ("roi_activation",   "mean"),
    )
    .round(2)
    .sort_values("total_opp_cost", ascending=False)
    .reset_index()
)
print()
print("Segment Summary:")
display(seg_summary)


## Section 6 — Publication-Quality Plots

In [ ]:
# ── Plot 1: Weibull PDF Fit ─────────────────────────────────────────────────
x_clip = np.percentile(x, 98)
x_plot = np.linspace(0.1, x_clip, 600)
pdf_v  = weibull_min.pdf(x_plot, shape, loc=0, scale=scale)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(x, bins=70, density=True, alpha=0.35, color=PALETTE["dark"],
        label="Observed data (repeat buyers)", range=(0, x_clip))
ax.plot(x_plot, pdf_v, color=PALETTE["Champions"], lw=2.5,
        label=f"Weibull MLE  β={shape:.2f}, λ={scale:.0f} days")
ax.axvline(theoretical_mean, color=PALETTE["accent"], lw=1.8, ls="--",
           label=f"E[T] = {theoretical_mean:.0f} days")
ax.set_xlabel("Average inter-purchase interval (days)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Weibull Distribution Fit — Olist Repeat Buyers",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("plots/plot1_weibull_fit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → plots/plot1_weibull_fit.png")


In [ ]:
# ── Plot 2: Survival & Hazard ───────────────────────────────────────────────
surv_v   = weibull_min.sf(x_plot,  shape, loc=0, scale=scale)
hazard_v = weibull_min.pdf(x_plot, shape, loc=0, scale=scale) / (surv_v + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival
ax = axes[0]
ax.plot(x_plot, surv_v, color=PALETTE["Nurture"], lw=2.5)
ax.fill_between(x_plot, surv_v, alpha=0.12, color=PALETTE["Nurture"])
milestones = [(30, PALETTE["Champions"]), (90, PALETTE["At Risk"]), (180, PALETTE["accent"])]
for t_mark, col in milestones:
    s = weibull_min.sf(t_mark, shape, loc=0, scale=scale)
    ax.axvline(t_mark, color=col, ls=":", lw=1.5)
    ax.annotate(f"{t_mark}d: {s:.0%} still inactive",
                xy=(t_mark, s), xytext=(t_mark + 10, s + 0.03),
                fontsize=8.5, color=col)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Days since last purchase", fontsize=11)
ax.set_ylabel("S(t) — Fraction not yet repurchased", fontsize=11)
ax.set_title(f"Survival Function $S(t)$", fontsize=12, fontweight="bold")

# Hazard
ax = axes[1]
ax.plot(x_plot, hazard_v, color=PALETTE["At Risk"], lw=2.5)
ax.fill_between(x_plot, hazard_v, alpha=0.12, color=PALETTE["At Risk"])
ax.annotate("Highest CRM\nimpact window",
            xy=(x_plot[5], hazard_v[5]),
            xytext=(80, hazard_v[5] * 1.35),
            fontsize=8.5, color=PALETTE["dark"],
            arrowprops=dict(arrowstyle="->", color=PALETTE["dark"]))
ax.set_xlabel("Days since last purchase", fontsize=11)
ax.set_ylabel("h(t) — instantaneous repurchase rate", fontsize=11)
ax.set_title(f"Hazard Function $h(t)$  [β={shape:.2f}]", fontsize=12, fontweight="bold")

plt.suptitle("Survival & Hazard Functions", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plots/plot2_survival_hazard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → plots/plot2_survival_hazard.png")


In [ ]:
# ── Plot 3: Conditional Repurchase Probability ─────────────────────────────
t0_range = np.linspace(1, 500, 500)
p_curve  = conditional_prob(t0_range, HORIZON)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t0_range, p_curve, color=PALETTE["dark"], lw=2.5)
ax.fill_between(t0_range, p_curve, alpha=0.08, color=PALETTE["dark"])

annotations = [
    (30,  PALETTE["Champions"]),
    (60,  PALETTE["Nurture"]),
    (90,  PALETTE["At Risk"]),
    (180, PALETTE["accent"]),
    (365, PALETTE["Dormant"]),
]
for days, color in annotations:
    p = float(conditional_prob(days, HORIZON))
    ax.scatter(days, p, color=color, zorder=5, s=60)
    ax.annotate(f"{days}d → {p:.0%}",
                xy=(days, p), xytext=(days + 12, p + 0.005),
                fontsize=8.5, color=color,
                arrowprops=dict(arrowstyle="->", color=color, lw=0.8))

ax.set_xlabel(f"Days since last purchase ($t_0$)", fontsize=11)
ax.set_ylabel(f"P(repurchase within {HORIZON}d | $t_0$)", fontsize=11)
ax.set_title("Conditional Repurchase Probability — The Bridge: Past → Future",
             fontsize=13, fontweight="bold", pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig("plots/plot3_conditional_prob.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → plots/plot3_conditional_prob.png")


In [ ]:
# ── Plot 4: CRM Prioritization Matrix ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

sample = all_customers.sample(min(2000, len(all_customers)), random_state=42)
for seg in ["Champions", "Nurture", "At Risk", "Dormant"]:
    sub = sample[sample["segment"] == seg]
    ax.scatter(sub["prob_30d"], sub["avg_ticket"],
               color=PALETTE[seg], alpha=0.5, s=18, label=seg)

ax.axvline(prob_med,   color="gray", ls="--", lw=1.2, alpha=0.7)
ax.axhline(ticket_med, color="gray", ls="--", lw=1.2, alpha=0.7)

ax.text(prob_med * 0.05, ticket_med * 1.5, "⚠  At Risk",
        fontsize=10, color=PALETTE["At Risk"], alpha=0.9)
ax.text(prob_med * 1.05, ticket_med * 1.5, "🎯 Champions",
        fontsize=10, color=PALETTE["Champions"], alpha=0.9)
ax.text(prob_med * 0.05, ticket_med * 0.3, "💤 Dormant",
        fontsize=10, color=PALETTE["Dormant"], alpha=0.9)
ax.text(prob_med * 1.05, ticket_med * 0.3, "📢 Nurture",
        fontsize=10, color=PALETTE["Nurture"], alpha=0.9)

ax.set_xlabel("P(repurchase within 30 days)", fontsize=11)
ax.set_ylabel("Average ticket (R$)", fontsize=11)
ax.set_title("CRM Prioritization Matrix\nP(repurchase) × Average Ticket",
             fontsize=13, fontweight="bold", pad=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=9, markerscale=2)
plt.tight_layout()
plt.savefig("plots/plot4_crm_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → plots/plot4_crm_matrix.png")


In [ ]:
# ── Plot 5: Opportunity Cost Distribution & ROI by Segment ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: OC histogram
ax = axes[0]
oc = all_customers["opportunity_cost"].clip(
    upper=np.percentile(all_customers["opportunity_cost"], 97)
)
ax.hist(oc, bins=55, color=PALETTE["At Risk"], alpha=0.7, edgecolor="white")
ax.axvline(oc.mean(), color=PALETTE["accent"], lw=2, ls="--",
           label=f"Mean: R${oc.mean():.1f}")
ax.axvline(ACTIVATION_COST, color=PALETTE["Champions"], lw=1.5, ls=":",
           label=f"Activation cost: R${ACTIVATION_COST:.0f}")
ax.set_xlabel("Opportunity cost per customer (R$)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Opportunity Cost Distribution\n(Cost of Doing Nothing)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

# Right: ROI boxplot by segment
ax = axes[1]
seg_order = ["Champions", "Nurture", "At Risk", "Dormant"]
roi_data  = [
    all_customers[all_customers["segment"] == s]["roi_activation"].dropna().values
    for s in seg_order
]
bp = ax.boxplot(roi_data, patch_artist=True,
                medianprops=dict(color="white", lw=2.5))
for patch, seg in zip(bp["boxes"], seg_order):
    patch.set_facecolor(PALETTE[seg])
    patch.set_alpha(0.75)
ax.axhline(0, color=PALETTE["accent"], lw=1.8, ls="--", label="Break-even (ROI = 0)")
ax.set_xticklabels(seg_order, fontsize=10)
ax.set_ylabel("Activation ROI", fontsize=11)
ax.set_title("ROI of Activation per Segment\n(Where to Invest CRM Budget)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

plt.suptitle("Opportunity Cost & Activation ROI", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plots/plot5_opp_cost_roi.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → plots/plot5_opp_cost_roi.png")


In [ ]:
# ── Plot 6: Lorenz Curve — Concentration of Opportunity Cost ───────────────
# Sort descending: top customers first (prioritisation view)
oc_sorted = all_customers["opportunity_cost"].sort_values(ascending=False).reset_index(drop=True)
oc_cumsum = oc_sorted.cumsum() / oc_sorted.sum()
pop_frac  = np.arange(1, len(oc_cumsum) + 1) / len(oc_cumsum)
equality  = np.linspace(0, 1, len(oc_cumsum))

# Gini (negative when curve is above diagonal — standard for descending sort)
gini = 1 - 2 * np.trapz(oc_cumsum.values, pop_frac)

idx_20  = int(0.20 * len(pop_frac))
pct_top = oc_cumsum.values[idx_20]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(pop_frac, oc_cumsum.values, color=PALETTE["Nurture"], lw=2.5,
        label=f"Opportunity cost  (Gini = {gini:.2f})")
ax.plot([0, 1], [0, 1], color="gray", ls="--", lw=1.2, label="Perfect equality")
ax.fill_between(pop_frac, oc_cumsum.values, equality, alpha=0.12, color=PALETTE["Nurture"])

ax.axvline(0.20,    color=PALETTE["At Risk"], ls=":", lw=1.5)
ax.axhline(pct_top, color=PALETTE["At Risk"], ls=":", lw=1.5)
ax.scatter(0.20, pct_top, color=PALETTE["At Risk"], zorder=5, s=70)
ax.annotate(
    f"Top 20% customers\n= {pct_top:.0%} of opportunity cost",
    xy=(0.20, pct_top), xytext=(0.28, pct_top - 0.15),
    fontsize=9, color=PALETTE["At Risk"],
    arrowprops=dict(arrowstyle="->", color=PALETTE["At Risk"]),
)
ax.set_xlabel("Fraction of customers (ranked by opportunity cost)", fontsize=11)
ax.set_ylabel("Cumulative share of opportunity cost", fontsize=11)
ax.set_title(f"Lorenz Curve — Opportunity Cost Concentration\nGini = {gini:.3f}",
             fontsize=13, fontweight="bold", pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("plots/plot6_lorenz.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → plots/plot6_lorenz.png")
print(f"Gini = {gini:.3f}  |  Top 20% customers = {pct_top:.1%} of total OC")


## Section 6b — Combined 6-Panel Figure

In [ ]:
# ── Combined 6-panel figure ─────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 13), facecolor="#F8FAFC")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.33)

# ① Weibull fit
ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor("#F8FAFC")
ax1.hist(x, bins=60, density=True, alpha=0.30, color=PALETTE["dark"], range=(0, x_clip))
ax1.plot(x_plot, pdf_v, color=PALETTE["Champions"], lw=2.2,
         label=f"Weibull  β={shape:.2f}, λ={scale:.0f}d")
ax1.axvline(theoretical_mean, color=PALETTE["accent"], lw=1.3, ls="--",
            label=f"E[T]={theoretical_mean:.0f}d")
ax1.set_title("① Weibull PDF Fit", fontsize=11, fontweight="bold")
ax1.set_xlabel("Avg interval (days)"); ax1.legend(fontsize=7.5)

# ② Survival
ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor("#F8FAFC")
ax2.plot(x_plot, surv_v, color=PALETTE["Nurture"], lw=2.2)
ax2.fill_between(x_plot, surv_v, alpha=0.10, color=PALETTE["Nurture"])
for t_m, col in [(30, PALETTE["Champions"]), (90, PALETTE["At Risk"]), (180, PALETTE["accent"])]:
    s = weibull_min.sf(t_m, shape, loc=0, scale=scale)
    ax2.axvline(t_m, color=col, ls=":", lw=1.2)
    ax2.annotate(f"{t_m}d:{s:.0%}", xy=(t_m, s), xytext=(t_m+8, s+0.04),
                 fontsize=7, color=col)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_title("② Survival Function S(t)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Days since last purchase")

# ③ Conditional prob
ax3 = fig.add_subplot(gs[0, 2]); ax3.set_facecolor("#F8FAFC")
ax3.plot(t0_range, p_curve, color=PALETTE["dark"], lw=2.2)
ax3.fill_between(t0_range, p_curve, alpha=0.07, color=PALETTE["dark"])
for days, col in [(30, PALETTE["Champions"]), (90, PALETTE["At Risk"]), (180, PALETTE["accent"])]:
    p = float(conditional_prob(days, HORIZON))
    ax3.axvline(days, color=col, ls="--", lw=1.1, alpha=0.8)
    ax3.annotate(f"{days}d:{p:.0%}", xy=(days, p), xytext=(days+8, p+0.005),
                 fontsize=7.5, color=col)
ax3.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax3.set_title("③ Conditional Repurchase Prob", fontsize=11, fontweight="bold")
ax3.set_xlabel("Days silent (t₀)")

# ④ CRM matrix
ax4 = fig.add_subplot(gs[1, 0]); ax4.set_facecolor("#F8FAFC")
for seg in ["Champions", "Nurture", "At Risk", "Dormant"]:
    sub = sample[sample["segment"] == seg]
    ax4.scatter(sub["prob_30d"], sub["avg_ticket"],
                color=PALETTE[seg], alpha=0.45, s=10, label=seg)
ax4.axvline(prob_med,   color="gray", ls="--", lw=1, alpha=0.6)
ax4.axhline(ticket_med, color="gray", ls="--", lw=1, alpha=0.6)
ax4.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax4.set_title("④ CRM Prioritization Matrix", fontsize=11, fontweight="bold")
ax4.set_xlabel("P(repurchase 30d)"); ax4.set_ylabel("Avg ticket (R$)")
ax4.legend(fontsize=7, markerscale=2)

# ⑤ OC distribution
ax5 = fig.add_subplot(gs[1, 1]); ax5.set_facecolor("#F8FAFC")
ax5.hist(oc, bins=50, color=PALETTE["At Risk"], alpha=0.7, edgecolor="white")
ax5.axvline(oc.mean(), color=PALETTE["accent"], lw=1.5, ls="--",
            label=f"Mean R${oc.mean():.1f}")
ax5.axvline(ACTIVATION_COST, color=PALETTE["Champions"], lw=1.3, ls=":",
            label=f"Cost R${ACTIVATION_COST:.0f}")
ax5.set_title("⑤ Opportunity Cost Distribution", fontsize=11, fontweight="bold")
ax5.set_xlabel("Opportunity cost (R$)"); ax5.legend(fontsize=7.5)

# ⑥ Lorenz
ax6 = fig.add_subplot(gs[1, 2]); ax6.set_facecolor("#F8FAFC")
ax6.plot(pop_frac, oc_cumsum.values, color=PALETTE["Nurture"], lw=2.2,
         label=f"Gini = {gini:.2f}")
ax6.plot([0, 1], [0, 1], color="gray", ls="--", lw=1.0, label="Equality")
ax6.fill_between(pop_frac, oc_cumsum.values, equality, alpha=0.10, color=PALETTE["Nurture"])
ax6.scatter(0.20, pct_top, color=PALETTE["At Risk"], zorder=5, s=50)
ax6.annotate(f"Top 20%\n= {pct_top:.0%} of OC",
             xy=(0.20, pct_top), xytext=(0.30, pct_top - 0.12),
             fontsize=7.5, color=PALETTE["At Risk"],
             arrowprops=dict(arrowstyle="->", color=PALETTE["At Risk"]))
ax6.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax6.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax6.set_title("⑥ Lorenz — OC Concentration", fontsize=11, fontweight="bold")
ax6.set_xlabel("Fraction of customers"); ax6.legend(fontsize=7.5)

fig.suptitle("Weibull Survival Analysis — Olist CRM Opportunity Cost",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plots/weibull_olist_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → plots/weibull_olist_final.png")


## Section 7 — Decision Table & Export

In [ ]:
OUTPUT_COLS = [
    "customer", "days_silent", "avg_ticket", "total_spent",
    "n_orders", "prob_30d", "expected_revenue",
    "expected_net_value", "opportunity_cost", "roi_activation",
    "segment", "action",
]

decision_table = (
    all_customers[OUTPUT_COLS]
    .sort_values("opportunity_cost", ascending=False)
    .reset_index(drop=True)
)
decision_table.to_csv("crm_priority_list.csv", index=False)
print(f"Exported {len(decision_table):,} customers → crm_priority_list.csv")
print()
print("Top 15 by opportunity cost:")
display(decision_table.head(15)[[
    "customer", "days_silent", "avg_ticket", "prob_30d",
    "opportunity_cost", "roi_activation", "segment", "action"
]].round(2))


In [ ]:
# ── Executive summary ──────────────────────────────────────────────────────
print("=" * 65)
print("EXECUTIVE SUMMARY")
print("=" * 65)
print(f"  Weibull β:              {shape:.4f}")
print(f"  Weibull λ:              {scale:.2f} days")
print(f"  E[T] (theoretical):     {theoretical_mean:.1f} days")
print()
print(f"  Customers scored:       {len(all_customers):,}  (100% of base)")
print(f"  Fit base (repeat):      {n_repeat:,}  ({100*n_repeat/n_total:.1f}%)")
print()
for seg in ["Champions", "At Risk", "Nurture", "Dormant"]:
    n  = (all_customers["segment"] == seg).sum()
    oc = all_customers.loc[all_customers["segment"] == seg, "opportunity_cost"].sum()
    print(f"  {seg:<12}  {n:>8,} customers   total OC = R$ {oc:>10,.0f}")
print()
print(f"  Gini (OC concentration):  {gini:.4f}")
print(f"  Top 20% customers hold:   {pct_top:.1%} of opportunity cost")
print("=" * 65)
